In [1]:
#import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import re
import math
from collections import Counter, defaultdict
%matplotlib inline
import random
import itertools
from sentence_transformers import SentenceTransformer, util

In [2]:
import gdown
import pandas as pd

file_id = "1kSQIo1kkDb53Mg4hbBehsEk57u9mjzkF"
url = f"https://drive.google.com/uc?id={file_id}"

df = pd.read_csv(url)
print(df.shape)
df.head()

(45466, 24)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_9468\2522543376.py:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


Hybrid

In [ ]:
user_id = 10

In [ ]:
# load mapping file đã làm ở mục bert score
mapping_bridge = pd.read_csv("movie_mapping.csv")

mapping_bridge["movie_id"] = mapping_bridge["movie_id"].astype(int)
mapping_bridge["id"] = mapping_bridge["id"].astype(str)

df_final["id"] = df_final["id"].astype(str)

In [ ]:
# MovieLens movie_id -> TMDB id
movieid_to_tmdb = dict(
    zip(
        mapping_bridge["movie_id"],
        mapping_bridge["id"]))

In [ ]:
# TMDB id -> df_final index
tmdb_to_dfidx = dict(
    zip(
        df_final["id"],
        df_final.index))

In [ ]:
# ALS index -> MovieLens movie_id
alsidx_to_movieid = dict(
    zip(movie_encoder.transform(movie_encoder.classes_),
        movie_encoder.classes_))

In [ ]:
# Lấy seed movie (phim cuối cùng user xem)
user_train_history = df_ratings_train[
    df_ratings_train["user_id"] == user_id
]

if user_train_history.empty:
    raise ValueError(
        f"User {user_id} has no training history."
    )

sample_seed_movie = str(
    user_train_history.iloc[-1]["movie_id"]
).strip()

print("User ID:", user_id)
print("Seed Movie ID:", sample_seed_movie)

User ID: 10
Seed Movie ID: 10377


In [ ]:
from sentence_transformers import util
from sklearn.preprocessing import MinMaxScaler

num_movies = len(df_final)

df_final_bert = df_final.copy()
df_final_bert["id"] = (df_final_bert["id"].astype(str).str.strip())

matched_indices = df_final_bert[df_final_bert["id"] == sample_seed_movie].index

if len(matched_indices) == 0:
    raise ValueError(
        f"Seed movie {sample_seed_movie} not found in df_final.")

# Cosine similarity với toàn bộ kho phim
seed_idx = matched_indices[0]
target_embedding = movie_embeddings[seed_idx]

bert_scores = util.cos_sim(
    target_embedding,
    movie_embeddings
)[0].cpu().numpy()

# Normalize BERT
bert_scores_scaled = MinMaxScaler().fit_transform(
    bert_scores.reshape(-1, 1)
).flatten()

In [ ]:
als_scores_ordered = np.zeros(num_movies)

try:
    user_idx = user_encoder.transform([user_id])[0]

except ValueError:
    raise ValueError(
        f"UserID {user_id} not found."
    )

user_items = user_item_matrix[user_idx]

ids, als_scores_raw = als_model.recommend(
    userid=user_idx,
    user_items=user_items,
    N=len(movie_encoder.classes_),
    filter_already_liked_items=False
)

for als_idx, score in zip(ids, als_scores_raw):

    # ALS index → MovieLens movie_id
    movie_id = alsidx_to_movieid.get(als_idx)

    if movie_id is None:
        continue

    # MovieLens movie_id → TMDB id
    tmdb_id = movieid_to_tmdb.get(movie_id)

    if tmdb_id is None:
        continue

    # TMDB id → df_final index
    df_idx = tmdb_to_dfidx.get(str(tmdb_id))

    if df_idx is None:
        continue

    als_scores_ordered[df_idx] = score

# Normalize ALS
als_scores_scaled = MinMaxScaler().fit_transform(
    als_scores_ordered.reshape(-1, 1)
).flatten()

In [ ]:
# hybrid
alpha = 0.7

hybrid_scores = (alpha * bert_scores_scaled+ (1 - alpha) * als_scores_scaled)

In [ ]:
# loại fim user đã xem
seen_movies = (
    df_ratings_cleaned[
        df_ratings_cleaned["user_id"] == user_id
    ]["movie_id"]
    .astype(int)
    .unique()
)

for movie_id in seen_movies:

    tmdb_id = movieid_to_tmdb.get(movie_id)

    if tmdb_id is None:
        continue

    df_idx = tmdb_to_dfidx.get(str(tmdb_id))

    if df_idx is not None:
        hybrid_scores[df_idx] = -1

In [ ]:
# top k
TOP_K = 10

top_indices = np.argsort(
    hybrid_scores
)[::-1][:TOP_K]

recommendations = (
    df_final.iloc[top_indices][
        ["id", "title"]
    ]
    .copy()
)

recommendations["Hybrid Score"] = (
    hybrid_scores[top_indices]
)

recommendations.reset_index(
    drop=True,
    inplace=True
)

print("===== HYBRID RECOMMENDATIONS =====")
print(f"User ID: {user_id}")
print(f"Seed Movie ID: {sample_seed_movie}")

recommendations

===== HYBRID RECOMMENDATIONS =====
User ID: 10
Seed Movie ID: 10377


,id,title,Hybrid Score
0,10377,My Cousin Vinny,0.944596
1,10276,What About Bob?,0.694996
2,380,Rain Man,0.692955
3,2619,Splash,0.687609
4,4011,Beetlejuice,0.686875
5,88,Dirty Dancing,0.681675
6,378,Raising Arizona,0.680757
7,2005,Sister Act,0.677523
8,251,Ghost,0.673592
9,620,Ghostbusters,0.673591
